# 05 V6 — CNN–SE–Feature Attention | CIC-IDS2017 Expanded Features (Kaggle)

**Mục tiêu:** Huấn luyện một bộ *ablation* CNN, CNN+SE, CNN+Feature Attention, CNN+SE+Feature Attention, Multi-Scale Residual CNN, A5 Hybrid trên **cùng feature variant và group-aware split** của Notebook **03B** (giống Notebook 04 V2). Chọn checkpoint / cấu hình dựa **chỉ trên validation**, đánh giá test sau khi lựa chọn.

**Dữ liệu:** `cic_expanded_splits.parquet`, `feature_sets.json`, `experiment_manifest.json` từ output Notebook 03B. `Label` là nhãn 15 lớp; `_split` phải là `train/val/test`. Metadata `_source_*`, `_duplicate_group`, nhãn không được phép làm đầu vào.

**Lưu ý khoa học:** Trục Conv1D ở đây là **trục feature**, KHÔNG có temporal attention. Grouping exact duplicates chỉ xử lý trùng vector, chưa tách độc lập host/session/time. Điểm số V6 so sánh được với Notebook 04 **V2** trên cùng split nhưng không so trực tiếp với CNN V1–V5. Split test đã được xem qua Notebook 04 V2; kết quả này vẫn là **exploratory pilot**, không phải benchmark hoàn toàn độc lập.

**Kaggle:** Add Input output Notebook 03B, bật GPU; không cần source code dataset riêng. Các model tự định nghĩa ngay trong notebook.

## 1. Thiết lập và GPU

In [ ]:
from pathlib import Path
import json, time, random, hashlib, gc
from dataclasses import asdict, dataclass
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import joblib

SEED=42
SEARCH_MODE='balanced'   # 'fast' (kiểm tra pipeline) | 'balanced' (khuyến nghị) | 'full' (nhiều GPU hours)
INPUT_DIR=None           # hoặc Path('/kaggle/input/<output-notebook-03b>/nids_outputs/03B_CIC_Feature_Engineering')
OUTPUT=Path('/kaggle/working/nids_outputs/05_CNN_Ablation_V6_Expanded') if Path('/kaggle/working').exists() else Path('reports/notebook_outputs/05_CNN_Ablation_V6_Expanded')
OUTPUT.mkdir(parents=True,exist_ok=True)
assert SEARCH_MODE in ('fast','balanced','full')
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cuda':
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
BUDGET={
 'fast':{'epochs':10,'patience':3,'second_seeds':[], 'extra_grid':False},
 'balanced':{'epochs':50,'patience':9,'second_seeds':[43], 'extra_grid':True},
 'full':{'epochs':85,'patience':14,'second_seeds':[43,44], 'extra_grid':True},
}[SEARCH_MODE]
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
set_seed(SEED)
print('device:',DEVICE,'torch:',torch.__version__,'budget:',BUDGET)
if DEVICE.type=='cuda':print('GPU:',torch.cuda.get_device_name(0))

## 2. Đọc artifact 03B, đối chiếu manifest, chống trùng dữ liệu giữa splits

In [ ]:
if INPUT_DIR is None:
    local=Path('/kaggle/working/nids_outputs/03B_CIC_Feature_Engineering')
    candidates=[local] if (local/'cic_expanded_splits.parquet').is_file() else []
    if not candidates and Path('/kaggle/input').exists():
        candidates=sorted({p.parent for p in Path('/kaggle/input').rglob('cic_expanded_splits.parquet')})
else:
    candidates=[Path(INPUT_DIR)]
needed=['cic_expanded_splits.parquet','feature_sets.json','experiment_manifest.json']
valid=[p for p in candidates if all((p/f).is_file() for f in needed)]
if len(valid)!=1:
    raise FileNotFoundError(f'Cần đúng một output 03B với {needed}, có {valid}. Add Input output Notebook 03B hoặc đặt INPUT_DIR cụ thể.')
input_dir=valid[0]
split_path=input_dir/'cic_expanded_splits.parquet'
source_manifest=json.loads((input_dir/'experiment_manifest.json').read_text(encoding='utf8'))
feature_sets=json.loads((input_dir/'feature_sets.json').read_text(encoding='utf8'))
feature_variant=source_manifest['validation_selected_feature_set']
if feature_variant not in feature_sets:raise ValueError(f'Unknown variant {feature_variant}')
features=list(feature_sets[feature_variant])
sample=pd.read_parquet(split_path)
required={'Label','_binary_label','_split','_source_file','_source_row','_duplicate_group'}
if not required.issubset(sample.columns):raise ValueError(f'Missing {required-set(sample.columns)}')
if not features or len(set(features))!=len(features) or not set(features).issubset(sample.columns):raise ValueError('Feature set invalid')
if any(x.startswith('_') or x=='Label' for x in features):raise ValueError('Metadata/label leakage')
if set(sample['_split'].astype(str))!={'train','val','test'}:raise ValueError('Expected train/val/test')
if len(sample)!=int(source_manifest['sample_rows']):raise ValueError('Row count mismatch manifest')
if sample.groupby('_duplicate_group')['_split'].nunique().max()>1:raise ValueError('Exact-feature duplicate leakage across splits')
if sample.duplicated(['_source_file','_source_row']).any():raise ValueError('Duplicate source rows')
labels=sample['Label'].astype(str)
if not np.array_equal(sample['_binary_label'].astype(int).to_numpy(),labels.ne('BENIGN').astype(int).to_numpy()):raise ValueError('Wrong binary labels')
idx={part:sample['_split'].eq(part).to_numpy() for part in ['train','val','test']}
encoder=LabelEncoder().fit(labels.loc[idx['train']])
if set(encoder.classes_)!=set(labels):raise ValueError('Rare class absent from train')
y={part:encoder.transform(labels.loc[mask]).astype(np.int64) for part,mask in idx.items()}
benign_id=int(encoder.transform(['BENIGN'])[0]);class_ids=np.arange(len(encoder.classes_))
raw=sample[features].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
sha=hashlib.sha256(split_path.read_bytes()).hexdigest()
print('03B input:',split_path,'SHA256:',sha)
print('feature variant:',feature_variant,'| features:',len(features),'| classes:',len(encoder.classes_))
print('splits:',{part:int(mask.sum()) for part,mask in idx.items()})
print(pd.crosstab(labels,sample['_split']).to_string())
(OUTPUT/'feature_names.json').write_text(json.dumps(features,ensure_ascii=False,indent=2),encoding='utf8')

## 3. Preprocessing TRAIN-ONLY — riêng với bộ features mới

In [ ]:
preprocess={};X={}
for name,scaler in [('standard',StandardScaler()),('robust',RobustScaler())]:
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    train_imputed=imputer.fit_transform(raw.loc[idx['train']])
    arrays={'train':scaler.fit_transform(train_imputed).astype(np.float32)}
    for part in ['val','test']:
        arrays[part]=scaler.transform(imputer.transform(raw.loc[idx[part]])).astype(np.float32)
    for part,data in arrays.items():
        if data.shape[1]!=len(features) or not np.isfinite(data).all():raise ValueError((name,part,'invalid transformed matrix'))
    preprocess[name]={'imputer':imputer,'scaler':scaler}
    X[name]=arrays
counts=np.bincount(y['train'],minlength=len(encoder.classes_))
print('TRAIN counts:',dict(zip(encoder.classes_.tolist(),counts.tolist())))
print('No fit on validation/test')

## 4. Kiến trúc: MLP + ablation CNN 1D + Multi-Scale Residual + A5 Hybrid

SE là *channel attention*, AttentionPool là *feature-position attention*. Cả hai không phải temporal modeling. Các mạng giữ nguyên code kiến trúc gốc từ V4, nhưng dùng `len(features)` mới (thường 60).

In [ ]:
class SEBlock(nn.Module):
    def __init__(self,c,reduction=8):
        super().__init__()
        self.net=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Flatten(),
            nn.Linear(c,max(4,c//reduction)),nn.ReLU(),nn.Linear(max(4,c//reduction),c),nn.Sigmoid())
    def forward(self,z):return z*self.net(z).unsqueeze(-1)

class AttentionPool(nn.Module):
    def __init__(self,c):
        super().__init__();self.score=nn.Sequential(nn.Linear(c,max(8,c//2)),nn.Tanh(),nn.Linear(max(8,c//2),1))
    def forward(self,z):
        h=z.transpose(1,2)
        w=torch.softmax(self.score(h).squeeze(-1),dim=1)
        return (h*w.unsqueeze(-1)).sum(dim=1)

class BasicFeatureCNN(nn.Module):
    def __init__(self,f,k,use_se=False,use_attn=False,width=64,dropout=.2):
        super().__init__();self.f=f
        self.conv=nn.Sequential(nn.Conv1d(1,width,3,padding=1),nn.ReLU(),nn.Conv1d(width,width,3,padding=1),nn.ReLU())
        self.se=SEBlock(width) if use_se else nn.Identity()
        self.pool=AttentionPool(width) if use_attn else nn.AdaptiveAvgPool1d(1)
        self.head=nn.Sequential(nn.Linear(width,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,k))
    def forward(self,x):
        z=self.se(self.conv(x.unsqueeze(1)))
        h=self.pool(z)
        if h.ndim==3:h=h.squeeze(-1)
        return self.head(h)

class ResConvBlock(nn.Module):
    def __init__(self,c,dropout):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(c,c,3,padding=1),nn.BatchNorm1d(c),nn.GELU(),
                nn.Conv1d(c,c,3,padding=1),nn.BatchNorm1d(c))
        self.drop=nn.Dropout1d(dropout)
    def forward(self,x):return torch.nn.functional.gelu(x+self.drop(self.net(x)))

class MultiScaleResidualCNN(nn.Module):
    def __init__(self,f,k,width=64,dropout=.2,use_se=True,use_attn=True,hybrid=False):
        super().__init__();self.f=f;self.hybrid=hybrid
        self.multi=nn.ModuleList([nn.Conv1d(1,width,sz,padding=sz//2) for sz in [1,3,5]])
        self.proj=nn.Sequential(nn.Conv1d(width*3,width,1),nn.BatchNorm1d(width),nn.GELU())
        self.blocks=nn.Sequential(ResConvBlock(width,dropout),ResConvBlock(width,dropout))
        self.se=SEBlock(width) if use_se else nn.Identity()
        self.pool=AttentionPool(width) if use_attn else nn.AdaptiveAvgPool1d(1)
        # Attention + avg/max retain both focused and global information.
        merged=width*3
        if hybrid:
            self.raw_path=nn.Sequential(nn.Linear(f,128),nn.LayerNorm(128),nn.GELU(),
                          nn.Dropout(dropout),nn.Linear(128,width),nn.GELU())
            merged+=width
        self.head=nn.Sequential(nn.Linear(merged,128),nn.GELU(),nn.Dropout(dropout),
                                nn.Linear(128,64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,k))
    def forward(self,x):
        z=self.proj(torch.cat([part(x.unsqueeze(1)) for part in self.multi],dim=1))
        z=self.se(self.blocks(z))
        h=self.pool(z)
        if h.ndim==3:h=h.squeeze(-1)
        pieces=[h,z.mean(-1),z.amax(-1)]
        if self.hybrid:pieces.append(self.raw_path(x))
        return self.head(torch.cat(pieces,dim=1))

class MLPBaseline(nn.Module):
    def __init__(self,f,k,dropout=.2,width=128):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(f,width),nn.ReLU(),nn.Dropout(dropout),
             nn.Linear(width,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,k))
    def forward(self,x):return self.net(x)

ARCHS=['MLP','A0_CNN','A1_CNN_SE','A2_CNN_FeatureAttn','A3_CNN_SE_FeatureAttn',
       'A4_MultiScaleRes_SE_Attn','A5_HybridMultiScaleRes_SE_Attn']
def build_model(arch,width=64,dropout=.2):
    f=len(features);k=len(encoder.classes_)
    if arch=='MLP':return MLPBaseline(f,k,dropout,width=max(128,width))
    if arch.startswith('A0_'):return BasicFeatureCNN(f,k,False,False,width,dropout)
    if arch.startswith('A1_'):return BasicFeatureCNN(f,k,True,False,width,dropout)
    if arch.startswith('A2_'):return BasicFeatureCNN(f,k,False,True,width,dropout)
    if arch.startswith('A3_'):return BasicFeatureCNN(f,k,True,True,width,dropout)
    if arch.startswith('A4_'):return MultiScaleResidualCNN(f,k,width,dropout,hybrid=False)
    if arch=='A5_HybridMultiScaleRes_SE_Attn':return MultiScaleResidualCNN(f,k,width,dropout,True,True,True)
    if arch=='A5_no_SE':return MultiScaleResidualCNN(f,k,width,dropout,False,True,True)
    if arch=='A5_no_Attn':return MultiScaleResidualCNN(f,k,width,dropout,True,False,True)
    if arch=='A5_no_Raw':return MultiScaleResidualCNN(f,k,width,dropout,True,True,False)
    if arch=='A5_backbone_only':return MultiScaleResidualCNN(f,k,width,dropout,False,False,False)
    raise ValueError(arch)

for a in ['MLP','A0_CNN','A1_CNN_SE','A2_CNN_FeatureAttn','A3_CNN_SE_FeatureAttn','A4_MultiScaleRes_SE_Attn','A5_HybridMultiScaleRes_SE_Attn','A5_no_Raw']:
    m=build_model(a)
    with torch.no_grad():assert m(torch.randn(2,len(features))).shape==(2,len(encoder.classes_))
    print(a, sum(p.numel() for p in m.parameters()),'parameters')

## 5. Training/validation: class-aware CE, AdamW, scheduler, early stopping

Lựa chọn best checkpoint theo **validation 15-class Macro-F1**, Accuracy dùng tie-break. Test không được dùng trong vòng thử cấu hình.

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class TrialConfig:
    arch:str
    scaler:str='standard'
    lr:float=0.001
    wd:float=0.0001
    dropout:float=0.2
    width:int=64
    weight_power:float=0.5
    loss:str='ce'
    batch_size:int=256
    seed:int=42

def loss_weights(power):
    if power==0:return torch.ones(len(counts),device=DEVICE)
    w=(counts.sum()/(len(counts)*np.maximum(counts,1)))**power
    w=np.clip(w,0.3,10.0)
    w=w/w.mean()
    return torch.as_tensor(w,dtype=torch.float32,device=DEVICE)

class FocalLoss(nn.Module):
    def __init__(self,weight,gamma=1.5):
        super().__init__();self.register_buffer('weights',weight);self.gamma=gamma
    def forward(self,logits,target):
        logp=torch.nn.functional.log_softmax(logits,dim=1)
        logpt=logp.gather(1,target[:,None]).squeeze(1)
        pt=logpt.exp();terms=-(1-pt).pow(self.gamma)*logpt
        return (terms*self.weights[target]).sum()/self.weights[target].sum().clamp_min(1.0)

def get_loss(cfg):
    weights=loss_weights(cfg.weight_power)
    if cfg.loss=='ce':return nn.CrossEntropyLoss(weight=weights)
    if cfg.loss=='focal':return FocalLoss(weights)
    raise ValueError(cfg.loss)

def get_loader(part,scaler_name,batch,seed):
    dataset=TensorDataset(torch.from_numpy(X[scaler_name][part]),torch.from_numpy(y[part]))
    return DataLoader(dataset,batch_size=batch,shuffle=(part=='train'),
            generator=torch.Generator().manual_seed(seed),pin_memory=(DEVICE.type=='cuda'),num_workers=0)

@torch.inference_mode()
def predict_logits(model,part,scaler_name,batch=1024):
    model.eval();result=[]
    for xx,_ in get_loader(part,scaler_name,batch,SEED):
        result.append(model(xx.to(DEVICE,non_blocking=True)).float().cpu().numpy())
    return np.concatenate(result,axis=0)

def val_scores(model,cfg):
    logit=predict_logits(model,'val',cfg.scaler)
    pred=logit.argmax(axis=1)
    truth=y['val']
    return (float(f1_score(truth,pred,labels=np.arange(len(encoder.classes_)),average='macro',zero_division=0)),
            float(accuracy_score(truth,pred)))

def run_trial(cfg,trial_id,max_epochs=None,patience=None):
    max_epochs=max_epochs or BUDGET['epochs'];patience=patience or BUDGET['patience']
    set_seed(cfg.seed)
    model=build_model(cfg.arch,cfg.width,cfg.dropout).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.lr,weight_decay=cfg.wd)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=.5,patience=4,min_lr=1e-5)
    criterion=get_loss(cfg)
    name=f'{trial_id:03d}_{cfg.arch}'
    folder=OUTPUT/'trials'/name;folder.mkdir(parents=True,exist_ok=True)
    best=(-1.,-1.);best_epoch=0;stale=0;history=[];start=time.perf_counter()
    for epoch in range(1,max_epochs+1):
        model.train();loss_sum=0.
        loader=get_loader('train',cfg.scaler,cfg.batch_size,cfg.seed+epoch)
        for xx,yy in loader:
            xx=xx.to(DEVICE,non_blocking=True);yy=yy.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(model(xx),yy)
            if not torch.isfinite(loss):raise FloatingPointError(f'Non-finite loss: {name}')
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),max_norm=2.)
            optimizer.step();loss_sum+=float(loss.item())*len(yy)
        f1,acc=val_scores(model,cfg)
        scheduler.step(f1)
        history.append({'trial_id':trial_id,'arch':cfg.arch,'epoch':epoch,
                         'train_loss':loss_sum/len(y['train']),'val_macro_f1':f1,
                         'val_accuracy':acc,'lr':optimizer.param_groups[0]['lr']})
        key=(f1,acc)
        if key>best:
            best=key;best_epoch=epoch;stale=0
            torch.save({'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                        'config':asdict(cfg),'best_epoch':epoch,'best_val_f1':f1,
                        'split_sha256':sha,'classes':encoder.classes_.tolist(),'features':features},folder/'best.pt')
        else:stale+=1
        if epoch==1 or epoch%10==0 or stale>=patience:
            print(f'{name} epoch={epoch} valF1={f1:.4f} best={best[0]:.4f} acc={acc:.4f}',flush=True)
        if stale>=patience:break
    torch.save({'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'config':asdict(cfg),'last_epoch':epoch,'split_sha256':sha},folder/'last.pt')
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    result={**asdict(cfg),'trial_id':trial_id,'epochs':epoch,'best_epoch':best_epoch,
            'best_val_macro_f1':best[0],'best_val_accuracy':best[1],
            'seconds':round(time.perf_counter()-start,1),'checkpoint':str(folder/'best.pt')}
    (folder/'metrics.json').write_text(json.dumps(result,indent=2,ensure_ascii=False))
    del model,optimizer,loader;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return result

## 6. Stage A — Ablation công bằng cùng cấu hình

Mỗi kiến trúc được huấn luyện một lần với cùng seed, split, scaler và ngân sách epoch. Các cấu hình được khóa trước khi xem test.

In [ ]:
ARCHITECTURES=['MLP','A0_CNN','A1_CNN_SE','A2_CNN_FeatureAttn','A3_CNN_SE_FeatureAttn','A4_MultiScaleRes_SE_Attn','A5_HybridMultiScaleRes_SE_Attn','A5_no_Raw']
results=[]
def run_and_store(cfg):
    trial_id=len(results)+1
    print(f'\nTRIAL {trial_id}: {cfg.arch}, seed={cfg.seed}, lr={cfg.lr}, scaler={cfg.scaler}',flush=True)
    r=run_trial(cfg,trial_id)
    results.append(r)
    pd.DataFrame(results).to_csv(OUTPUT/'all_validation_trials.csv',index=False)
    return r
for arch in ARCHITECTURES:
    run_and_store(TrialConfig(arch=arch,scaler='standard',lr=.0015,wd=.0001,
                              dropout=.10,width=96,weight_power=.35,loss='ce',batch_size=256,seed=SEED))
trials=pd.DataFrame(results)
print(trials.sort_values('best_val_macro_f1',ascending=False)[['arch','best_val_macro_f1','best_val_accuracy','best_epoch']].to_string(index=False))

## 7. Stage B — Search hạn chế cho các CNN mạnh (chỉ validation)

Chỉ chạy khi `SEARCH_MODE` là balanced/full. Tìm kiếm nhỏ trên learning rate, regularization, scaler và weight power. Nếu GPU chậm, đặt `SEARCH_MODE="fast"` và Run All từ đầu; không sửa cấu hình dựa trên test.

In [ ]:
if BUDGET['extra_grid']:
    search_configs=[
        dict(scaler='standard',lr=.0007,wd=.0005,dropout=.20,width=96,weight_power=.20),
        dict(scaler='robust',lr=.0010,wd=.0001,dropout=.10,width=96,weight_power=.35),
        dict(scaler='standard',lr=.0010,wd=.0001,dropout=.15,width=96,weight_power=.50),
        dict(scaler='standard',lr=.0015,wd=.0002,dropout=.10,width=128,weight_power=.25),
    ]
    for arch in ['A5_HybridMultiScaleRes_SE_Attn','A5_no_Raw']:
        for h in search_configs:
            run_and_store(TrialConfig(arch=arch,batch_size=256,seed=SEED,loss='ce',**h))
print('Number of trials:',len(results))

## 8. Stage C — Repeated seeds cho cấu hình tốt nhất trên validation

Chọn best trial từ Stage A+B, chạy thêm seeds để đánh giá độ ổn định mà không xem test. Thực nghiệm này dùng **cùng split**, không phải independent resampling.

In [ ]:
trial_table=pd.DataFrame(results).sort_values(['best_val_macro_f1','best_val_accuracy'],ascending=False).reset_index(drop=True)
best=trial_table.iloc[0]
print('Best validation trial before extra seeds:',best[['arch','seed','best_val_macro_f1','best_val_accuracy']].to_dict())
for sd in BUDGET['second_seeds']:
    params={key:best[key] for key in ['arch','scaler','lr','wd','dropout','width','weight_power','loss','batch_size']}
    params['width']=int(params['width']);params['batch_size']=int(params['batch_size']);params['seed']=int(sd)
    run_and_store(TrialConfig(**params))
trials=pd.DataFrame(results)
summary=trials.groupby('arch',as_index=False).agg(val_macro_f1_mean=('best_val_macro_f1','mean'),val_macro_f1_std=('best_val_macro_f1','std'),n_trials=('trial_id','count'))
summary.to_csv(OUTPUT/'architecture_validation_summary.csv',index=False)
display(summary.sort_values('val_macro_f1_mean',ascending=False))

## 9. Validation-only selection và optional BENIGN bias

Ưu tiên **Macro-F1** (không bắt buộc FPR dưới ngưỡng); xuất Pareto của Macro-F1 và FPR. Chỉ dùng validation để chọn model/offset. Bias giúp điều chỉnh cảnh báo giả nhưng có thể giảm khả năng phân loại XSS/Brute Force.

In [ ]:
@torch.inference_mode()
def load_logits(row,part):
    ck=torch.load(row['checkpoint'],map_location='cpu',weights_only=False)
    cfg=ck['config']
    model=build_model(cfg['arch'],int(cfg['width']),float(cfg['dropout'])).to(DEVICE)
    model.load_state_dict(ck['state_dict'])
    z=predict_logits(model,part,cfg['scaler'])
    del model,ck;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
    return z

def biased_predict(logits,bias):
    z=logits.copy();z[:,benign_id]+=float(bias)
    return z.argmax(1)
def evaluate(truth,pred):
    tr=np.asarray(truth)!=benign_id;pr=np.asarray(pred)!=benign_id
    tn=int(((~tr)&(~pr)).sum());fp=int(((~tr)&pr).sum())
    return {'accuracy':float(accuracy_score(truth,pred)),
            'macro_f1':float(f1_score(truth,pred,labels=class_ids,average='macro',zero_division=0)),
            'weighted_f1':float(f1_score(truth,pred,labels=class_ids,average='weighted',zero_division=0)),
            'attack_recall':float(recall_score(tr,pr,zero_division=0)),
            'attack_precision':float(precision_score(tr,pr,zero_division=0)),
            'fpr':float(fp/(tn+fp)) if tn+fp else float('nan')}
BIAS_GRID=[-0.5,0.,0.25,0.5,0.75,1.,1.5]
choices=[]
for _,row in trials.iterrows():
    logits=load_logits(row,'val')
    for bias in BIAS_GRID:
        record={'trial_id':int(row['trial_id']),'arch':row['arch'],'seed':int(row['seed']),
                'bias':float(bias),'checkpoint':row['checkpoint']}
        record.update(evaluate(y['val'],biased_predict(logits,bias)))
        choices.append(record)
validation_grid=pd.DataFrame(choices).sort_values(['macro_f1','accuracy'],ascending=False)
validation_grid.drop(columns='checkpoint').to_csv(OUTPUT/'validation_bias_selection_grid.csv',index=False)
selection=validation_grid.iloc[0].to_dict()
(OUTPUT/'validation_selection.json').write_text(json.dumps(selection,indent=2,ensure_ascii=False,default=str),encoding='utf8')
print('Selected by validation Macro-F1 only:',selection)
print('Validation FPR tradeoff (top candidates):')
display(validation_grid[['arch','seed','bias','macro_f1','accuracy','fpr','attack_recall']].head(12))

## 10. Test evaluation cuối cùng sau khi khóa lựa chọn

**Chỉ model đã chọn** được kiểm thử. Không dùng số liệu test để đổi loss, features, bias hoặc model; test của protocol 03B vốn đã có baseline nên vẫn là pilot.

In [ ]:
selected_row=trials.loc[trials.trial_id.eq(int(selection['trial_id']))].iloc[0]
selected_logits=load_logits(selected_row,'test')
test_truth=y['test']
test_raw_pred=biased_predict(selected_logits,0)
test_final_pred=biased_predict(selected_logits,float(selection['bias']))
final_metrics={
 'model':str(selection['arch']),'seed':int(selection['seed']),'bias':float(selection['bias']),
 'selected_by':'validation 15-class macro-F1 (accuracy tie-break)',
 'validation_macro_f1':float(selection['macro_f1']),
 'test_raw':evaluate(test_truth,test_raw_pred),
 'test_selected':evaluate(test_truth,test_final_pred),
 'n_features':len(features),'feature_variant':feature_variant,
 'n_test':int(len(test_truth)),'source_split_sha256':sha,
 'protocol':'EXPLORATORY_GROUP_EXACT_DUPLICATES_NOT_HOST_TIME_DISJOINT',
 'warning':'Same test protocol as 04 V2, but already inspected there. For final claims obtain untouched external holdout.'}
(OUTPUT/'final_test_metrics.json').write_text(json.dumps(final_metrics,indent=2,ensure_ascii=False),encoding='utf8')
print(json.dumps(final_metrics,indent=2,ensure_ascii=False))
rep=classification_report(test_truth,test_final_pred,labels=class_ids,target_names=encoder.classes_,output_dict=True,zero_division=0)
(OUTPUT/'test_classification_report.json').write_text(json.dumps(rep,indent=2,ensure_ascii=False),encoding='utf8')
pd.DataFrame(rep).T.to_csv(OUTPUT/'test_classification_report.csv')
cm=confusion_matrix(test_truth,test_final_pred,labels=class_ids)
np.save(OUTPUT/'test_confusion_matrix.npy',cm)
pred_df=pd.DataFrame({'true_label':encoder.inverse_transform(test_truth),
                      'pred_label':encoder.inverse_transform(test_final_pred),
                      'source_file':sample.loc[idx['test'],'_source_file'].to_numpy(),
                      'source_row':sample.loc[idx['test'],'_source_row'].to_numpy()})
pred_df.to_csv(OUTPUT/'test_predictions.csv',index=False)
wrong=pred_df[pred_df.true_label.ne(pred_df.pred_label)]
wrong.groupby(['true_label','pred_label']).size().reset_index(name='count').sort_values('count',ascending=False).to_csv(OUTPUT/'test_confusion_pairs.csv',index=False)
joblib.dump({'imputer':preprocess[selected_row['scaler']]['imputer'],
             'scaler':preprocess[selected_row['scaler']]['scaler'],
             'label_encoder':encoder,'features':features,'feature_variant':feature_variant,
             'split_sha256':sha},OUTPUT/'selected_preprocessing.joblib')
# Copy best checkpoint to a predictable filename without overwriting V1-V5
from shutil import copyfile
copyfile(str(selected_row['checkpoint']),str(OUTPUT/'selected_best.pt'))

## 11. So sánh đúng protocol với Notebook 04 V2 và vẽ biểu đồ

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig,ax=plt.subplots(figsize=(12,5))
ordered=summary.sort_values('val_macro_f1_mean',ascending=False)
ax.bar(ordered.arch,ordered.val_macro_f1_mean,yerr=ordered.val_macro_f1_std.fillna(0),capsize=3)
ax.set_ylim(0,1);ax.set_title('V6 architecture validation Macro-F1 (multiple trial configs, not matched-seed means)')
ax.tick_params(axis='x',rotation=45);ax.set_ylabel('Macro-F1');plt.tight_layout()
fig.savefig(OUTPUT/'validation_architecture_summary.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(12,10))
ConfusionMatrixDisplay(cm,display_labels=encoder.classes_).plot(ax=ax,xticks_rotation=90,colorbar=False)
ax.set_title('V6 selected model: 15-class test confusion matrix');plt.tight_layout()
fig.savefig(OUTPUT/'selected_test_confusion_matrix.png',dpi=160);plt.show()
# Optional actual artifacts from Notebook 04 V2. Do not silently compare different splits.
baseline_candidates=sorted(Path('/kaggle/input').rglob('test_metrics.csv')) if Path('/kaggle/input').exists() else []
print('Baseline 04 V2 reference (previously observed on SAME 03B protocol): LightGBM macro-F1=0.9064, accuracy=0.9785.')
print('For formal comparison, attach 04 V2 output and verify its manifest split_sha256 ==',sha)
print('Found test_metrics.csv candidates:',baseline_candidates[:5])
manifest={'experiment':'05_CNN_Ablation_V6_Expanded','protocol':source_manifest.get('protocol'),
 'input_03b_manifest':source_manifest,'split_sha256':sha,'split_file':str(split_path),
 'features':features,'feature_variant':feature_variant,'n_classes':len(encoder.classes_),
 'n_rows':len(sample),'search_mode':SEARCH_MODE,'budget':BUDGET,
 'model_selection':'validation_macro_f1','selected':selection,
 'limitations':['No IP/time/session-disjoint evaluation','All architectures use feature-position, not temporal attention',
                'Rare class support makes per-class F1 unstable','Test exposed in 04 V2, exploratory only']}
(OUTPUT/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False,default=str),encoding='utf8')
print('Saved',len(list(OUTPUT.rglob('*'))),'items under',OUTPUT)

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/nids_outputs/03B_CIC_Feature_Engineering")

print("03B exists:", p.exists())

if p.exists():
    for f in p.iterdir():
        print(f.name)